# Session 8 — Building and Deploying ML-Powered Web Applications using Flask and AWS SageMaker

**Goal:** split an ML application into the two pieces a real deployment usually has —
a managed **model-hosting endpoint** (AWS SageMaker) and a lightweight **web
front-end** (Flask) that calls it — instead of loading the model directly inside the
web server process (Session 6's pattern).

## Why separate the model from the web app?

Loading a model directly inside your Flask process (Session 6) is simple but couples
your web app's scaling to your model's resource needs — a traffic spike now also
needs more GPU/CPU for inference. SageMaker hosts the model as its own autoscaling
endpoint; the Flask app becomes a thin client that just calls it over HTTP, which is
how most production ML web apps are actually built.

## Prerequisites

This session needs an **AWS account** with SageMaker access — not available in this
sandbox. The training/deployment cells show complete, correct `boto3`/`sagemaker` SDK
code; the Flask app cells (which only need `requests` + `flask`, no AWS) are written
to run against whatever endpoint URL you configure.

```bash
pip install boto3 sagemaker flask
aws configure   # needs an IAM user/role with SageMaker permissions
```

In [ ]:
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker import get_execution_role

REGION = "us-east-1"
BUCKET = "your-sagemaker-bucket"
ROLE = get_execution_role()  # or a hardcoded IAM role ARN outside of SageMaker Studio

## Step 1 — Train a scikit-learn model via a SageMaker training job

SageMaker's `SKLearn` estimator runs your training script inside a managed container,
reading data from S3 and writing the trained model back to S3 — no manual instance
management.

In [ ]:
train_script = '''\
import argparse, os, joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    args = parser.parse_args()

    df = pd.read_csv(os.path.join(args.train, "train.csv"))
    X, y = df.drop(columns="target"), df["target"]

    model = RandomForestClassifier(n_estimators=args.n_estimators, random_state=0)
    model.fit(X, y)
    joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))
'''
with open("sagemaker_train.py", "w") as f:
    f.write(train_script)
print(train_script)

In [ ]:
estimator = SKLearn(
    entry_point="sagemaker_train.py",
    role=ROLE,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    hyperparameters={"n-estimators": 200},
)

estimator.fit({"train": f"s3://{BUCKET}/heart-disease/train.csv"})
print("Training job complete. Model artifact:", estimator.model_data)

## Step 2 — Deploy to a real-time SageMaker endpoint

`.deploy()` provisions a managed, autoscaling HTTPS endpoint that hosts the model —
directly comparable to the Vertex AI `.deploy()` call in Session 4, on the other
cloud.

In [ ]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="heart-disease-endpoint",
)
print(f"Endpoint deployed: {predictor.endpoint_name}")

## Step 3 — Call the endpoint directly (sanity check)

In [ ]:
import numpy as np

sample = np.array([[58, 1, 0, 128, 216, 0, 0, 131, 1, 2.2, 1, 3, 3]])
result = predictor.predict(sample)
print("Direct SageMaker prediction:", result)

## Step 4 — The Flask web app that calls the endpoint

This is the part that needs no AWS credentials to *write* (only to run against a real
endpoint) — a normal Flask app whose `/predict` route forwards to SageMaker via
`boto3` instead of running a model in-process.

In [ ]:
flask_app = '''\
from flask import Flask, request, jsonify, render_template_string
import boto3
import json

app = Flask(__name__)
runtime = boto3.client("sagemaker-runtime", region_name="us-east-1")
ENDPOINT_NAME = "heart-disease-endpoint"

FORM_HTML = \'\'\'
<form method="POST" action="/predict">
  <input name="features" placeholder="comma-separated feature values">
  <button type="submit">Predict</button>
</form>
\'\'\'

@app.route("/")
def index():
    return render_template_string(FORM_HTML)

@app.route("/predict", methods=["POST"])
def predict():
    features = [float(x) for x in request.form["features"].split(",")]
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="text/csv",
        Body=",".join(map(str, features)),
    )
    prediction = response["Body"].read().decode()
    return jsonify({"prediction": prediction})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
'''
with open("session8_flask_app.py", "w") as f:
    f.write(flask_app)
print(flask_app)

## Step 5 — Clean up

SageMaker endpoints bill per hour while running, same as Vertex AI (Session 4) —
always delete when you're done.

In [ ]:
predictor.delete_endpoint()
print("Endpoint deleted -- billing stopped.")

## What to try next

* Put the Flask app itself behind an Application Load Balancer on ECS/Fargate, so the
  web tier also autoscales independently of the SageMaker endpoint.
* Compare this "web app calls managed endpoint" split against Session 6's "everything
  in one container" pattern — the right choice depends on whether your model needs
  GPU/expensive hardware that the web tier doesn't.
* Session 9 replaces the hand-written training script here with SageMaker Autopilot,
  removing the need to write `sagemaker_train.py` at all.